In [ ]:
from llms.langchain_tutorial.deepseek import get_local_embeddings

embeddings_model = get_local_embeddings()

In [ ]:
texts = [
        "你好吗",
        "你的名字是什么",
        "我的肚子好痛啊",
        "肠胃不舒服",
        "我在吃东西"
    ]
embeddings = embeddings_model.embed_documents(texts)

len(embeddings), len(embeddings[0])

In [ ]:
embeddings[0][:10]

# cosine similarity

In [ ]:
import numpy as np

def normalize(x):
    x = np.asarray(x)
    norms = np.sum(np.multiply(x, x))
    norms = np.sqrt(norms)
    return x / norms

for i in range(5):
    similarity = np.dot(normalize(embeddings[2]), normalize(embeddings[i]))
    print(f'"{texts[2]}"与"{texts[i]}"的语义相似度为：{similarity}')

# cache
## LocalFileStore

In [ ]:
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model, store, namespace=embeddings_model.model_name
)


In [ ]:
%%time
cached_embedder.embed_documents(texts)

In [ ]:
%%time
cached_embedder.embed_documents(texts)

In [ ]:
list(store.yield_keys())

## InMemoryByteStore

In [ ]:
from langchain.storage import InMemoryByteStore

store = InMemoryByteStore()

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model, store, namespace=embeddings_model.model_name
)

cached_embedder.embed_documents

## RedisStore（可选）
Redis 缓存需要另行部署服务；上面的本地文件和内存缓存示例可直接运行。

## 使用本地向量检索结果并由 DeepSeek 解释

In [ ]:
import numpy as np
from llms.langchain_tutorial.deepseek import get_deepseek_chat

query = '身体不舒服'
query_vector = np.asarray(embeddings_model.embed_query(query))
document_vectors = np.asarray(embeddings)
scores = document_vectors @ query_vector / (
    np.linalg.norm(document_vectors, axis=1) * np.linalg.norm(query_vector)
)
nearest_text = texts[int(np.argmax(scores))]
response = get_deepseek_chat().invoke(
    f'检索词是“{query}”，本地向量模型选出的相似句子是“{nearest_text}”。请简要解释两者的语义联系。'
)
print(response.content)